# View 3: Nonprofit mix in high-need vs. low-need areas (Massachusetts)
Which types of nonprofits exist in high-poverty vs. low-poverty tracts and towns?
Uses View 1 outputs (tracts_by_type.csv, towns_by_type.csv) so both views count the same organizations.

## Config

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SAVE_DIR_NAME = "view3_nonprofit_mix"

TRACT_FILE = "tracts_by_type.csv"   # View 1 output
TOWN_FILE = "towns_by_type.csv"     # View 1 output

NEED_VAR = "poverty_rate_pct"       # same need measure as View 1 (client's choice)
SPLIT = 1 / 4                       # top / bottom quarter, same as View 1
LOW_POP = 500                       # areas under this are left out, same as View 1
STATE_FIPS = "25"

# Same NTEE grouping as View 1 / ntee_groups.py
CATEGORIES = ["Human services", "Health", "Education", "Arts & culture", "Environment & animals",
              "Recreation & sports", "Religion", "Community & philanthropy", "Other"]
UNKNOWN = "Unknown (no NTEE code)"
MAP_CATEGORIES = [c for c in CATEGORIES if c != "Other"]

LOWPOP = "Low population (excluded)"
NEED_COLORS = {"High need": "#D9822B", "Middle": "#C9C9C9", "Low need": "#2F6690", LOWPOP: "#EDEDED"}

## Setup

In [ ]:
import subprocess

if IN_COLAB:
    from google.colab import files
    UP = Path("/content/upload")
    UP.mkdir(exist_ok=True)
    missing = [f for f in [TRACT_FILE, TOWN_FILE] if not (UP / f).exists()]
    if missing:
        print("Choose these files from your computer:", ", ".join(missing))
        for fname, data in files.upload().items():
            (UP / fname).write_bytes(data)
    DATA_DIR = UP
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas", "folium"], check=True)
else:
    DATA_DIR = Path("data")

SAVE_DIR = (Path("/content") if IN_COLAB else Path(".")) / SAVE_DIR_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)


def slug(name: str) -> str:
    """Same as View 1: 'Arts & culture' -> 'arts_culture'."""
    s = "".join(ch if ch.isalnum() else "_" for ch in name.lower())
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_")


CAT_COLS = {c: f"n_{slug(c)}" for c in CATEGORIES + [UNKNOWN]}

## 1. Load View 1 tract and town tables

In [ ]:
t = pd.read_csv(DATA_DIR / TRACT_FILE, dtype={"geoid": str, "county_fips": str})
t["geoid"] = t.geoid.str.zfill(11)
w = pd.read_csv(DATA_DIR / TOWN_FILE, dtype={"town_geoid": str})
w["town_geoid"] = w.town_geoid.str.zfill(10)

for df, name in [(t, "tracts"), (w, "towns")]:
    miss = [c for c in list(CAT_COLS.values()) + ["n_all_nonprofits", "population", NEED_VAR] if c not in df]
    assert not miss, f"{name}: missing columns {miss}"

print(f"{len(t):,} tracts | {len(w):,} towns")
print(f"Organizations in tracts: {int(t.n_all_nonprofits.sum()):,}  (View 1 README: 22,006)")

## 2. Split areas into high-need and low-need (poverty quarters)

In [ ]:
def need_groups(df: pd.DataFrame, unit: str) -> pd.DataFrame:
    ok = (df.population >= LOW_POP) & df[NEED_VAR].notna()
    hi = df.loc[ok, NEED_VAR].quantile(1 - SPLIT)
    lo = df.loc[ok, NEED_VAR].quantile(SPLIT)
    g = np.select([df[NEED_VAR] >= hi, df[NEED_VAR] <= lo], ["High need", "Low need"], "Middle")
    df["need_group"] = np.where(ok, g, LOWPOP)
    print(f"{unit}: High need = poverty >= {hi:.1f}% | Low need = poverty <= {lo:.1f}%")
    print(df.need_group.value_counts().to_string(), "\n")
    return df


t = need_groups(t, "Tracts")
w = need_groups(w, "Towns")

## 3. Compare the nonprofit mix: share and per 1,000 residents

In [ ]:
def mix_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for g in ["High need", "Middle", "Low need"]:
        d = df[df.need_group == g]
        pop, total = d.population.sum(), d.n_all_nonprofits.sum()
        known = sum(d[CAT_COLS[c]].sum() for c in CATEGORIES)
        for c in CATEGORIES + [UNKNOWN]:
            n = d[CAT_COLS[c]].sum()
            rows.append({"need_group": g, "category": c, "organizations": int(n),
                         "residents": int(pop),
                         "per_1k_residents": round(1000 * n / pop, 2),
                         "share_of_known_pct": round(100 * n / known, 1) if c != UNKNOWN else np.nan,
                         "share_of_all_pct": round(100 * n / total, 1)})
    return pd.DataFrame(rows)


tract_mix = mix_summary(t)
town_mix = mix_summary(w)

cols = ["High need", "Low need"]
print("TRACTS: share of organizations with an NTEE code (%)")
print(tract_mix.pivot(index="category", columns="need_group", values="share_of_known_pct")
      .reindex(CATEGORIES)[cols].to_string())
print("\nTRACTS: organizations per 1,000 residents")
print(tract_mix.pivot(index="category", columns="need_group", values="per_1k_residents")
      .reindex(CATEGORIES + [UNKNOWN])[cols].to_string())

## 3b. Share of areas with at least one nonprofit of each type
Pooled per-1,000 rates are pulled up by a few downtown tracts where many headquarters are registered
(one Boston tract has 731 organizations). Counting areas that have at least one organization of a type
is not affected by those outliers.

In [ ]:
def presence_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for g in ["High need", "Middle", "Low need"]:
        d = df[df.need_group == g]
        for c in ["All nonprofits"] + CATEGORIES:
            col = "n_all_nonprofits" if c == "All nonprofits" else CAT_COLS[c]
            rows.append({"need_group": g, "category": c, "areas": len(d),
                         "areas_with_any": int((d[col] > 0).sum()),
                         "pct_areas_with_any": round(100 * (d[col] > 0).mean(), 1)})
    return pd.DataFrame(rows)


tract_presence = presence_summary(t)
town_presence = presence_summary(w)
print("TRACTS: % of tracts with at least one nonprofit of this type")
print(tract_presence.pivot(index="category", columns="need_group", values="pct_areas_with_any")
      .reindex(["All nonprofits"] + CATEGORIES)[cols].to_string())

## 4. Check: results without college-area tracts

In [ ]:
college = t.college_area_flag.astype(str).str.lower().eq("true")
print("College-area tracts in High need:", int((college & (t.need_group == "High need")).sum()))
check = mix_summary(t[~college]).pivot(index="category", columns="need_group",
                                       values="share_of_known_pct").reindex(CATEGORIES)[cols]
print("\nShare (%) excluding college-area tracts:\n", check.to_string())

## 5. Static charts

In [ ]:
import matplotlib.pyplot as plt


def bar_chart(mix, value, xlabel, title, fname):
    p = mix[mix.category != UNKNOWN].pivot(index="category", columns="need_group", values=value).reindex(CATEGORIES)
    fig, ax = plt.subplots(figsize=(10, 6))
    y, h = np.arange(len(p)), 0.38
    ax.barh(y - h / 2, p["High need"], h, color=NEED_COLORS["High need"], label="High need (top quarter poverty)")
    ax.barh(y + h / 2, p["Low need"], h, color=NEED_COLORS["Low need"], label="Low need (bottom quarter poverty)")
    ax.set_yticks(y, p.index)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=13)
    ax.legend(frameon=False, loc="lower right")
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    fig.text(0.01, -0.03, "Nonprofits are placed at their filing address. About a quarter have no NTEE code "
             "and are not in any category.", fontsize=9, color="#555555")
    fig.savefig(SAVE_DIR / fname, dpi=200, bbox_inches="tight")
    plt.show()


bar_chart(tract_mix, "share_of_known_pct", "Share of nonprofits with an NTEE code (%)",
          "Nonprofit mix: high-need vs. low-need tracts, Massachusetts", "view3_mix_share_tracts.png")
bar_chart(tract_presence[tract_presence.category != "All nonprofits"], "pct_areas_with_any",
          "Tracts with at least one nonprofit of this type (%)",
          "Which nonprofit types are present: high-need vs. low-need tracts", "view3_presence_tracts.png")
bar_chart(town_mix, "share_of_known_pct", "Share of nonprofits with an NTEE code (%)",
          "Nonprofit mix: high-need vs. low-need towns, Massachusetts", "view3_mix_share_towns.png")

## 6. Tract boundaries

In [ ]:
import io, zipfile, requests
import geopandas as gpd

CACHE = Path("/content/geo_cache") if IN_COLAB else Path("geo_cache")
CACHE.mkdir(parents=True, exist_ok=True)


def fetch(urls):
    for url in urls:
        folder = CACHE / Path(url).stem
        try:
            if not folder.exists():
                r = requests.get(url, timeout=600)
                r.raise_for_status()
                zipfile.ZipFile(io.BytesIO(r.content)).extractall(folder)
            return gpd.read_file(next(folder.glob("*.shp")))
        except Exception as e:
            print("Failed:", url, e)
    raise RuntimeError("Could not download boundaries")


tracts = fetch([f"https://www2.census.gov/geo/tiger/GENZ{y}/shp/cb_{y}_{STATE_FIPS}_tract_500k.zip"
                for y in (2024, 2023, 2022, 2020)])[["GEOID", "geometry"]]
print(f"Tract boundaries: {len(tracts):,}")

## 7. Interactive map

In [ ]:
import folium

TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
ATTR = "Tiles &copy; Esri &mdash; Esri, HERE, Garmin, OpenStreetMap contributors"
SHARE_EDGES = [10, 20, 30, 40]
SHARE_COLORS = ["#F6F1E1", "#D6E2EE", "#A3BED8", "#6D94BC", "#2F6690"]   # View 1 blue scale

t["n_known"] = t[[CAT_COLS[c] for c in CATEGORIES]].sum(axis=1)
for c in CATEGORIES:
    t[f"share_{slug(c)}"] = np.where(t.n_known > 0, (100 * t[CAT_COLS[c]] / t.n_known).round(1), np.nan)

gt = tracts.merge(t, left_on="GEOID", right_on="geoid", how="inner")
gt["geometry"] = gt.geometry.simplify(0.0003)
gt["college_area_flag"] = gt.college_area_flag.astype(str)
tip_fields = ["tract_name", "population", NEED_VAR, "need_group", "n_all_nonprofits", "n_known",
              "share_human_services", "share_health", "share_education", "share_arts_culture"]
tip_alias = ["Tract", "Residents", "Poverty rate (%)", "Need group", "All nonprofits", "With NTEE code",
             "Human services (%)", "Health (%)", "Education (%)", "Arts & culture (%)"]
gj = gt[tip_fields + ["geometry"]].copy()

m = folium.Map(location=[42.25, -71.8], zoom_start=8, tiles=TILES, attr=ATTR)


def add_layer(name, colors, show):
    g = gj.assign(color=colors)
    fg = folium.FeatureGroup(name=name, overlay=False, show=show)
    folium.GeoJson(g.to_json(), style_function=lambda f: {"fillColor": f["properties"]["color"],
                   "color": "#FFFFFF", "weight": 0.2, "fillOpacity": 0.8},
                   tooltip=folium.GeoJsonTooltip(fields=tip_fields, aliases=tip_alias)).add_to(fg)
    fg.add_to(m)


add_layer("Need group (poverty quarters)", gt.need_group.map(NEED_COLORS).values, True)
for c in MAP_CATEGORIES:
    v = gt[f"share_{slug(c)}"]
    col = np.where(v.isna(), NEED_COLORS[LOWPOP], np.array(SHARE_COLORS)[np.searchsorted(SHARE_EDGES, v.fillna(0))])
    add_layer(f"Share: {c}", col, False)

folium.LayerControl(collapsed=False).add_to(m)
m.save(str(SAVE_DIR / "view3_tracts.html"))
m

## 8. Save

In [ ]:
share_cols = [f"share_{slug(c)}" for c in CATEGORIES]
out_t = t.rename(columns={"geoid": "final_tract_geoid"})[
    ["final_tract_geoid", "tract_name", "population", NEED_VAR, "median_household_income",
     "college_area_flag", "need_group", "n_all_nonprofits", "n_known"] + list(CAT_COLS.values()) + share_cols]
out_t.to_csv(SAVE_DIR / "view3_tracts.csv", index=False)

w["n_known"] = w[[CAT_COLS[c] for c in CATEGORIES]].sum(axis=1)
for c in CATEGORIES:
    w[f"share_{slug(c)}"] = np.where(w.n_known > 0, (100 * w[CAT_COLS[c]] / w.n_known).round(1), np.nan)
w[["town_geoid", "town", "population", NEED_VAR, "median_household_income", "college_area_flag",
   "need_group", "n_all_nonprofits", "n_known"] + list(CAT_COLS.values()) + share_cols].to_csv(
    SAVE_DIR / "view3_towns.csv", index=False)

tract_mix.to_csv(SAVE_DIR / "view3_mix_summary_tracts.csv", index=False)
town_mix.to_csv(SAVE_DIR / "view3_mix_summary_towns.csv", index=False)
tract_presence.to_csv(SAVE_DIR / "view3_presence_summary_tracts.csv", index=False)
town_presence.to_csv(SAVE_DIR / "view3_presence_summary_towns.csv", index=False)

for p in sorted(SAVE_DIR.iterdir()):
    print(f"{p.name:40s} {p.stat().st_size / 1e3:9.1f} KB")

if IN_COLAB:
    import shutil
    files.download(shutil.make_archive(f"/content/{SAVE_DIR_NAME}", "zip", SAVE_DIR))